
# 01 — Cross-Attention, from Scratch

**Goal:** by the end of this notebook you can (1) explain cross-attention precisely to an
interviewer, (2) implement it correctly in PyTorch without looking anything up, and
(3) answer follow-up questions about how it shows up in real VLMs (Flamingo-style gated
cross-attention, Q-Formers, perceiver resamplers) and how it relates to the LLaVA-style
"just concatenate the vision tokens" approach used in models like Qwen2.5-VL.

Structure of every notebook in this series:
1. **Lesson** (read this, it's the part an interviewer expects you to already know)
2. **Implementation** (you write code — TODOs are marked, tests tell you if you're right)
3. **Quiz** (conceptual questions, answer in the markdown cell provided — no peeking below)
4. **Final Answers & Explanations** (full reference solution + quiz answers)

Don't scroll past the quiz until you've actually tried to answer it.



## 1. Lesson

### 1.1 Self-attention vs. cross-attention — the one-sentence distinction

In **self-attention**, Q, K, and V are all linear projections of the *same* sequence.
In **cross-attention**, Q comes from one sequence (the "querying" sequence) and K, V come
from a *different* sequence (the "context" sequence). That's the entire difference — the
math is identical, only where the tensors come from changes.

Formally, for a query sequence of length $L_q$ and a key/value sequence of length $L_{kv}$
(these need **not** be equal — this is the whole point of cross-attention):

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right) V
$$

- $Q \in \mathbb{R}^{L_q \times d_k}$ — projected from the query sequence
- $K \in \mathbb{R}^{L_{kv} \times d_k}$, $V \in \mathbb{R}^{L_{kv} \times d_v}$ — projected from the context sequence
- $M$ — an additive mask (usually just a padding mask here, see 1.3)
- Output shape: $L_q \times d_v$ — **one output row per query token**, each a weighted mixture
  of the context sequence's values.

### 1.2 Where this shows up

| Model family | Query sequence | Key/Value sequence |
|---|---|---|
| Original Transformer (encoder-decoder, e.g. machine translation) | decoder hidden states | encoder output |
| Flamingo / OpenFlamingo (VLM) | text token hidden states | vision encoder patch embeddings |
| BLIP-2 Q-Former | learned query tokens | frozen vision encoder features |
| Perceiver / Perceiver IO | small set of learned latents | raw high-dim input (pixels, audio, etc.) |
| LLaVA, Qwen2.5-VL (in contrast) | — | — *(no cross-attention: vision tokens are projected and concatenated into the same sequence the LLM self-attends over)* |

This last row matters for interviews: **not every VLM uses cross-attention**. LLaVA-style
models turn image patches into "pseudo-text" tokens via an MLP projector and prepend/interleave
them into the LLM's input sequence — the LLM then uses ordinary causal *self*-attention over
text+vision tokens together. Flamingo-style models keep the LLM's self-attention untouched and
instead insert *extra* cross-attention layers that let text tokens look at vision features
without changing the LLM's input sequence length. Trade-off: LLaVA-style is simpler and lets
the LLM's existing self-attention do multimodal fusion "for free," but vision tokens consume
context length and scale quadratically with everything else; Flamingo-style keeps context
length fixed regardless of image count but adds new parameters and a training recipe for them.

### 1.3 Masking in cross-attention

Causal masking (`i` can't attend to `j > i`) is a *self*-attention concept tied to
autoregressive generation over one sequence. Cross-attention's query and key/value sequences
are different sequences, so "causality between them" usually isn't meaningful — a text token
generated at decode step 5 is still allowed to attend to *all* vision patches, including ones
"after" it spatially, because there's no notion of vision-patch order corresponding to
generation order. What you *do* still need is a **padding mask**: if you batch variable-length
context sequences (e.g., different numbers of image patches, or padded encoder outputs), you
must mask out the padding positions in $K,V$ so they don't receive attention weight.

### 1.4 Multi-head cross-attention

Exactly like multi-head self-attention: split $d_{model}$ into $h$ heads of size
$d_k = d_{model}/h$, run scaled dot-product attention independently per head, concatenate,
project back with $W_O$. The only difference from multi-head self-attention is (again) that
$Q$'s projection input and $K,V$'s projection input come from different tensors.

### 1.5 Complexity

$O(L_q \cdot L_{kv} \cdot d)$ for the attention matrix, vs. $O(L^2 d)$ for self-attention where
$L_q = L_{kv} = L$. This matters in Perceiver-style architectures: by making $L_q$ a small
fixed number of learned latents (say 64) instead of the full input length, cross-attention
cost becomes *linear* in the size of the context sequence rather than quadratic — this is
precisely why Perceiver/Q-Former architectures can afford to cross-attend into thousands of
raw pixel or patch tokens.

### 1.6 Gated cross-attention (Flamingo)

Flamingo inserts cross-attention layers into a *frozen, pretrained* LLM. Naively adding a new
randomly-initialized layer into a frozen network would wreck its pretrained behavior at the
start of training. Flamingo's fix: gate the cross-attention (and the FFN that follows it) with
$\tanh(\alpha)$, where $\alpha$ is a learned scalar **initialized to 0**. At initialization,
$\tanh(0) = 0$, so the new layer contributes nothing and the model behaves exactly like the
original frozen LLM; $\alpha$ is then learned, letting the model gradually "turn on" the visual
information as training makes it useful. This is a general trick worth knowing: *zero-init a
gate on any new component you're splicing into a pretrained network.*


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)



## 2. Implementation

Implement `MultiHeadCrossAttention` below. Fill in every `# TODO`. Signature:

- `forward(query, key_value, key_padding_mask=None)`
  - `query`: `(batch, Lq, embed_dim)`
  - `key_value`: `(batch, Lkv, embed_dim)`
  - `key_padding_mask`: `(batch, Lkv)` boolean, `True` at positions that ARE padding (to be masked out)
  - returns `(batch, Lq, embed_dim)`

Use separate `nn.Linear` projections for Q, K, V, and a final output projection `out_proj`.


In [ ]:

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # TODO: define self.q_proj, self.k_proj, self.v_proj, self.out_proj
        # all nn.Linear(embed_dim, embed_dim, bias=True)
        self.q_proj = None
        self.k_proj = None
        self.v_proj = None
        self.out_proj = None

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, embed_dim) -> (batch, num_heads, seq, head_dim)
        # TODO
        raise NotImplementedError

    def forward(self, query: torch.Tensor, key_value: torch.Tensor,
                key_padding_mask: torch.Tensor = None) -> torch.Tensor:
        batch, Lq, _ = query.shape
        _, Lkv, _ = key_value.shape

        # TODO: project query -> Q, key_value -> K, key_value -> V
        Q = None
        K = None
        V = None

        # TODO: split into heads -> shapes (batch, num_heads, Lq/Lkv, head_dim)
        Q = None
        K = None
        V = None

        # TODO: scaled dot-product scores, shape (batch, num_heads, Lq, Lkv)
        scores = None

        if key_padding_mask is not None:
            # key_padding_mask: (batch, Lkv), True = pad position, should get -inf score
            # TODO: broadcast the mask to (batch, 1, 1, Lkv) and apply with masked_fill
            pass

        # TODO: softmax over the last dim (Lkv), then weighted sum with V
        attn_weights = None
        out = None  # (batch, num_heads, Lq, head_dim)

        # TODO: merge heads back to (batch, Lq, embed_dim), then out_proj
        out = None

        return out



### Sanity tests

Run these. They check shapes, masking behavior, and (crucially) numerical correctness against
PyTorch's own `nn.MultiheadAttention` with weights copied over, so you can trust your
implementation is not just "shaped right" but actually *correct*.


In [ ]:

# --- Test 1: output shape ---
embed_dim, num_heads = 32, 4
batch, Lq, Lkv = 2, 5, 7

mha = MultiHeadCrossAttention(embed_dim, num_heads)
query = torch.randn(batch, Lq, embed_dim)
kv = torch.randn(batch, Lkv, embed_dim)

out = mha(query, kv)
assert out.shape == (batch, Lq, embed_dim), f"got {out.shape}"
print("Test 1 passed: output shape", out.shape)


In [ ]:

# --- Test 2: padding mask actually zeroes out attention to padded positions ---
# Trick: if we mask out all but one KV position, every query row's output must equal
# out_proj(V_head_concat) for that single unmasked position (attention weight 1.0 on it).
mha2 = MultiHeadCrossAttention(embed_dim, num_heads)
query2 = torch.randn(1, 3, embed_dim)
kv2 = torch.randn(1, Lkv, embed_dim)

mask = torch.ones(1, Lkv, dtype=torch.bool)  # True = pad (masked out)
mask[0, 2] = False  # only position 2 is real

out2 = mha2(query2, kv2, key_padding_mask=mask)

# manually compute expected: attention collapses to a copy of V at position 2, projected
with torch.no_grad():
    V_full = mha2.v_proj(kv2)  # (1, Lkv, embed_dim)
    V_at_2 = V_full[:, 2:3, :].expand(-1, 3, -1)  # (1, 3, embed_dim), already in "merged head" layout
    expected = mha2.out_proj(V_at_2)

assert torch.allclose(out2, expected, atol=1e-5), "masking test failed"
print("Test 2 passed: padding mask correctly forces attention onto the unmasked position")


In [ ]:

# --- Test 3: numerical correctness against torch.nn.MultiheadAttention ---
ref = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
mine = MultiHeadCrossAttention(embed_dim, num_heads)

# Copy ref's combined in_proj_weight/bias into our separate q/k/v projections.
with torch.no_grad():
    w_q, w_k, w_v = ref.in_proj_weight.chunk(3, dim=0)
    b_q, b_k, b_v = ref.in_proj_bias.chunk(3, dim=0)
    mine.q_proj.weight.copy_(w_q); mine.q_proj.bias.copy_(b_q)
    mine.k_proj.weight.copy_(w_k); mine.k_proj.bias.copy_(b_k)
    mine.v_proj.weight.copy_(w_v); mine.v_proj.bias.copy_(b_v)
    mine.out_proj.weight.copy_(ref.out_proj.weight)
    mine.out_proj.bias.copy_(ref.out_proj.bias)

query3 = torch.randn(batch, Lq, embed_dim)
kv3 = torch.randn(batch, Lkv, embed_dim)
pad_mask = torch.zeros(batch, Lkv, dtype=torch.bool)
pad_mask[0, -2:] = True  # first batch element has 2 padded KV positions

ref_out, _ = ref(query3, kv3, kv3, key_padding_mask=pad_mask)
mine_out = mine(query3, kv3, key_padding_mask=pad_mask)

assert torch.allclose(ref_out, mine_out, atol=1e-4), \
    f"mismatch, max diff = {(ref_out - mine_out).abs().max().item()}"
print("Test 3 passed: matches nn.MultiheadAttention exactly (max diff "
      f"{(ref_out - mine_out).abs().max().item():.2e})")



## 3. Quiz

Answer each of these in your own words in the markdown cell below (or out loud, as if to an
interviewer) *before* looking at the answers in section 4.

1. What distinguishes cross-attention from self-attention in terms of tensor shapes, and why
   does that make $L_q \ne L_{kv}$ possible?
2. Why do we typically *not* apply a causal mask in cross-attention layers connecting a vision
   encoder to an LLM decoder, when we clearly *do* need one in the LLM's own self-attention?
3. In a VLM like Qwen2.5-VL, vision tokens are concatenated into the LLM's input sequence
   rather than consumed via cross-attention. What does this cost you as image resolution / the
   number of images grows, and what does the Flamingo-style alternative cost you instead?
4. What's the computational complexity of cross-attention in terms of $L_q$, $L_{kv}$, $d$, and
   why does making $L_q$ small and fixed (as in Perceiver Resampler / Q-Former) change the
   *scaling* with respect to input size, not just the constant factor?
5. Why is a key-padding mask often necessary for the key/value side of cross-attention (e.g.
   batches with different numbers of image patches) even when it might not be needed for
   single-example self-attention?
6. Explain gated cross-attention (Flamingo): what problem does the $\tanh(\alpha)$ gate solve,
   and why is $\alpha$ initialized to 0 specifically (not some other small value)?
7. During autoregressive generation, how would you cache computation in a cross-attention layer
   where K/V come from a frozen vision encoder that only needs to run once? Which of Q, K, V
   change at every decode step and which don't?

*(Your answers here)*



## 4. Final Answers & Explanations

### Q1 — Shape distinction
In self-attention, $Q, K, V$ are all linear projections of the *same* input tensor, so
necessarily $L_q = L_k = L_v = L$. In cross-attention, $Q$ is projected from a different tensor
than $K$ and $V$ are, so their sequence lengths are independent — $Q \in \mathbb{R}^{L_q \times d_k}$
comes from one sequence's length, $K, V \in \mathbb{R}^{L_{kv} \times \cdot}$ come from another's.
The attention matrix $QK^\top$ is $L_q \times L_{kv}$ — rectangular, not necessarily square. This
is *the* structural fact that makes cross-attention useful: you can query a long, fixed context
(e.g. all of an image's patches) with a short, variable-length query sequence (e.g. generated
text tokens), or vice versa (a handful of learned latents querying thousands of pixels, as in
Perceiver).

### Q2 — No causal mask needed
Causal masking exists to preserve the autoregressive property *within a single sequence*: token
$i$ must not see token $j > i$ because at generation time token $j$ doesn't exist yet. In
cross-attention between text and vision, the "future" doesn't apply to the vision side at all —
the entire image is available at once, there's no temporal/generation order over image patches
relative to the text being generated. So every text query token is allowed (and expected) to
attend to every vision patch, "future" or not, because there is no such thing as "future" image
content in this context. (The LLM's *own* self-attention over the text tokens it has generated
so far still needs a causal mask — that's a separate layer with a separate mask.)

### Q3 — Cost comparison
LLaVA/Qwen2.5-VL-style concatenation costs **context length**: every image patch becomes a
token the LLM's self-attention must include, so both the KV cache size and the $O(L^2)$
self-attention cost grow with image resolution and image count — a handful of high-res images
can consume most of the context window before any text is even generated. The upside is
simplicity (no new attention mechanism, the LLM's existing self-attention does the fusion) and
it tends to transfer well from strong pretrained decoder-only LLMs.
Flamingo-style cross-attention costs **new parameters and a nontrivial training recipe**
(gated cross-attention layers interleaved into a frozen LLM, needing careful init as in Q6) and
architectural complexity, but keeps the LLM's own sequence length independent of image content —
you can feed arbitrarily many/large images without inflating the text-side context window,
since images are consumed through cross-attention rather than occupying sequence positions.

### Q4 — Complexity and Perceiver-style scaling
Cross-attention costs $O(L_q \cdot L_{kv} \cdot d)$ for computing and applying the attention
matrix. In ordinary self-attention $L_q = L_{kv} = L$ so cost is $O(L^2 d)$ — quadratic in input
size. If instead $L_q = m$ is a small, *fixed* number of learned latent queries (independent of
input size) and only $L_{kv} = L$ grows with the input (e.g. thousands of raw pixel/patch
tokens), cost becomes $O(m \cdot L \cdot d)$ — **linear** in $L$, since $m$ is a constant. That's
not just a smaller constant factor, it's a different growth rate entirely, which is exactly why
Perceiver/Q-Former architectures can afford to cross-attend directly into very long raw inputs
where a same-sized self-attention block would be computationally infeasible.

### Q5 — Padding masks
Within one self-attention example there's typically no padding to worry about (real tokens
throughout), but when you *batch* multiple examples whose context sequences have different
natural lengths — e.g. images with different numbers of patches, or variable-length encoder
outputs — you must pad the batch to a common $L_{kv}$ to form a rectangular tensor. Without a
mask, the model would attend to and average in contentless padding vectors, corrupting the
output for every query token. The padding mask (applied as $-\infty$ to those positions' scores
before softmax) guarantees zero attention weight lands on pad positions regardless of what
values happen to sit there.

### Q6 — Gated cross-attention
Flamingo splices *new*, randomly-initialized cross-attention (and FFN) layers into an otherwise
frozen, already-pretrained LLM. If those new layers contributed their full (random,
untrained) output from step one, they'd immediately perturb the frozen LLM's carefully
pretrained activations and destroy its language modeling ability before any useful visual
signal had been learned. The fix is to scale each new layer's output by $\tanh(\alpha)$ with a
learned scalar $\alpha$ **initialized to exactly 0**: $\tanh(0) = 0$, so at initialization the
new layer's contribution is *exactly* zero and the wrapped model is functionally identical to
the original frozen LLM. Training then gradually pushes $\alpha$ away from 0 only insofar as
the visual signal actually helps, so the model can smoothly "dial in" cross-modal information
rather than being shocked by it. Any other small-but-nonzero init would still inject an
untrained perturbation on day one; zero is the only value that guarantees no behavior change
at $t=0$.

### Q7 — Caching during generation
$K$ and $V$ in this layer are projections of the *vision* features, which don't change across
decode steps — so they should be computed **once**, immediately after the vision encoder runs,
and cached/reused for every subsequent decode step (no recomputation, no growing cache — it's a
fixed-size cache equal to the number of vision tokens). $Q$, by contrast, is a projection of the
*current* text query token(s) and is genuinely new at every decode step, so it must be recomputed
each step (though typically it's just a single new token's worth of $Q$, since previous steps'
queries never need to be revisited in cross-attention — unlike self-attention's KV cache, there's
no "cross-attention Q cache" because old queries are simply discarded after use).
